# vLLM PD 分离架构指标实时曲线 v6

解析 vLLM PD 分离部署的日志文件（适配 `prefill_P0.log` 格式），展示 **10 个指标**：

前 7 个来自 vLLM Engine 周期打印行；后 3 个（NPU Hit Rate、CPU Hit Rate、CPU Stored）
来自 `scheduler.py` 每请求完成行（`cache stats: {...}` 格式）。

| 指标 | 来源 |
|---|---|
| Prefix Cache Hit Rate | Engine 周期行 |
| External Prefix Cache Hit Rate | Engine 周期行 |
| GPU KV Cache Usage | Engine 周期行 |
| Avg Prompt Throughput | Engine 周期行 |
| Avg Gen Throughput | Engine 周期行 |
| Running Requests | Engine 周期行 |
| Waiting Requests | Engine 周期行 |
| NPU Hit Rate per-req (%) | scheduler.py 每请求行 |
| CPU Hit Rate per-req (%) | scheduler.py 每请求行 |
| CPU Stored Blocks per-req | scheduler.py 每请求行 |

> **使用方法：** 按顺序运行各单元格；scheduler 指标（浅蓝背景）为散点图。

In [ ]:
import os

# ═══════════════════════════════════════════════
#  配置区：按需修改以下参数
# ═══════════════════════════════════════════════
LOG_SUBPATH = 'logs/0327'   # 相对 workspace 根目录的路径，或绝对路径
# ═══════════════════════════════════════════════

# notebook 位于 <workspace>/pd_disaggregation/analysis/
# 向上两级得到 workspace 根目录
_here = os.path.abspath(os.getcwd())
_ws   = os.path.dirname(os.path.dirname(_here))

LOG_DIR = (LOG_SUBPATH if os.path.isabs(LOG_SUBPATH)
           else os.path.join(_ws, LOG_SUBPATH))
LOG_DIR = os.path.abspath(LOG_DIR)

print(f'LOG_DIR : {LOG_DIR}  (exists={os.path.exists(LOG_DIR)})')

In [ ]:
%matplotlib widget

import os, re, ast, datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import ipywidgets as widgets
from IPython.display import display

print('OK')

In [ ]:
# ── 指标定义 ──────────────────────────────────────────────────────────────
# (列名, 子图标题, y轴范围或None, 数据源: 'engine'=Engine周期行 / 'sched'=scheduler每请求行)
METRICS = [
    ('hit_rate',          'Prefix Cache Hit Rate (%)',          (0, 100), 'engine'),
    ('ext_hit_rate',      'External Prefix Cache Hit Rate (%)', (0, 100), 'engine'),
    ('kv_cache',          'GPU KV Cache Usage (%)',             (0, 100), 'engine'),
    ('prompt_throughput', 'Avg Prompt Throughput (tok/s)',       None,    'engine'),
    ('gen_throughput',    'Avg Gen Throughput (tok/s)',          None,    'engine'),
    ('running',           'Running Requests',                    None,    'engine'),
    ('waiting',           'Waiting Requests',                    None,    'engine'),
    ('npu_hit_rate',      'NPU Hit Rate per-req (%)',           (0, 100), 'sched'),
    ('cpu_hit_rate',      'CPU Hit Rate per-req (%)',           (0, 100), 'sched'),
    ('cpu_stored',        'CPU Stored Blocks per-req',           None,    'sched'),
]

COLORS = [
    '#C0392B', '#1A5276', '#1E8449', '#6C3483',
    '#D35400', '#784212', '#0E6655', '#2C3E50',
    '#8B0000', '#00008B', '#4B0082', '#B8860B',
]

_ANSI_RE = re.compile(r'\x1b\[[0-9;]*m')


# ── 解析函数 ────────────────────────────────────────────────────────────────
def _parse_logs():
    """返回 (d_engine, d_sched, roles).

    d_engine[role]: DataFrame，来自 Engine 周期打印行
    d_sched[role] : DataFrame，来自 scheduler.py 每请求完成行
    """
    log_files = sorted(
        f for f in os.listdir(LOG_DIR)
        if (f.startswith('prefill') or f.startswith('decode')) and f.endswith('.log')
    )
    if not log_files:
        raise FileNotFoundError(f'LOG_DIR 中无 prefill*/decode*.log: {LOG_DIR}')

    d_engine, d_sched = {}, {}
    roles = []
    for fname in log_files:
        role = fname.replace('.log', '')
        roles.append(role)
        eng, sched = [], []

        with open(os.path.join(LOG_DIR, fname), encoding='utf-8', errors='replace') as fh:
            for line in fh:
                clean = _ANSI_RE.sub('', line)

                # ── Engine 周期指标行 ─────────────────────────────────────────
                if 'Prefix cache hit rate:' in clean:
                    ts_m      = re.search(r'(\d{2}-\d{2} \d{2}:\d{2}:\d{2})', clean)
                    rate_m    = re.search(r'(?<!External )Prefix cache hit rate: ([0-9.]+)%', clean)
                    ext_m     = re.search(r'External prefix cache hit rate: ([0-9.]+)%', clean)
                    gen_m     = re.search(r'Avg generation throughput: ([0-9.]+) tokens/s', clean)
                    kv_m      = re.search(r'GPU KV cache usage: ([0-9.]+)%', clean)
                    prompt_m  = re.search(r'Avg prompt throughput: ([0-9.]+) tokens/s', clean)
                    running_m = re.search(r'Running: ([0-9]+) reqs', clean)
                    waiting_m = re.search(r'Waiting: ([0-9]+) reqs', clean)
                    if ts_m and rate_m:
                        eng.append({
                            'timestamp':         f'2026-{ts_m.group(1)}',
                            'hit_rate':          float(rate_m.group(1)),
                            'ext_hit_rate':      float(ext_m.group(1))    if ext_m      else None,
                            'gen_throughput':    float(gen_m.group(1))    if gen_m      else None,
                            'kv_cache':          float(kv_m.group(1))     if kv_m       else None,
                            'prompt_throughput': float(prompt_m.group(1)) if prompt_m   else None,
                            'running':           int(running_m.group(1))  if running_m  else None,
                            'waiting':           int(waiting_m.group(1))  if waiting_m  else None,
                        })

                # ── scheduler.py 每请求完成行 ────────────────────────────────
                elif 'scheduler.py' in clean and 'cache stats:' in clean:
                    ts_m    = re.search(r'(\d{2}-\d{2} \d{2}:\d{2}:\d{2})', clean)
                    stats_m = re.search(r'cache stats: (\{[^}]+\})', clean)
                    if ts_m and stats_m:
                        try:
                            stats = ast.literal_eval(stats_m.group(1))
                            sched.append({
                                'timestamp':    f'2026-{ts_m.group(1)}',
                                'npu_hit_rate': round(stats['npu_hit_rate'] * 100, 2),
                                'cpu_hit_rate': round(stats['cpu_hit_rate'] * 100, 2),
                                'cpu_stored':   int(stats['cpu_stored']),
                            })
                        except Exception:
                            pass

        d_engine[role] = pd.DataFrame(eng)
        d_sched[role]  = pd.DataFrame(sched)

    return d_engine, d_sched, roles


# ── Figure（5 行 × 2 列 = 10 子图）──────────────────────────────────────────
plt.ioff()
fig, axes = plt.subplots(5, 2, figsize=(14, 16))
fig.subplots_adjust(hspace=0.45, wspace=0.28)
fig.suptitle('vLLM PD Metrics v6', fontsize=14, fontweight='bold')

_ax_list = []
for idx, (col, title, y_range, src) in enumerate(METRICS):
    ax = axes[idx // 2][idx % 2]
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('Time (s)', fontsize=8)
    ax.grid(True, alpha=0.3)
    if y_range:
        ax.set_ylim(y_range)
    if src == 'sched':
        ax.set_facecolor('#f0f4ff')   # 调度器指标浅蓝背景
    _ax_list.append((ax, col, y_range, src))

fig.tight_layout(rect=[0, 0, 1, 0.96])

_lines        = {}   # {(role, metric_idx): Line2D}
_visible      = {}
_role_buttons = {}
_legend_box   = widgets.HBox([], layout=widgets.Layout(flex_flow='row wrap'))


def _make_role_button(role, role_idx):
    btn = widgets.ToggleButton(
        value=True, description=role,
        layout=widgets.Layout(width='120px', height='28px'),
        style={'font_weight': 'bold'},
    )
    def _on_toggle(change, _role=role):
        vis = change['new']
        _visible[_role] = vis
        for mi in range(len(METRICS)):
            if (_role, mi) in _lines:
                _lines[(_role, mi)].set_visible(vis)
        for mi, (ax, *_rest) in enumerate(_ax_list):
            vl = [l for l in ax.get_lines() if l.get_visible()]
            if vl:
                ax.legend(handles=vl, labels=[l.get_label() for l in vl],
                          fontsize=7, loc='upper right')
            elif ax.get_legend():
                ax.get_legend().remove()
        fig.canvas.draw_idle()
    btn.observe(_on_toggle, names='value')
    return btn


def _sync_legend_buttons(roles):
    changed = False
    for ri, role in enumerate(roles):
        if role not in _role_buttons:
            _visible[role] = True
            _role_buttons[role] = _make_role_button(role, ri)
            changed = True
    stale = [r for r in _role_buttons if r not in set(roles)]
    for r in stale:
        del _role_buttons[r]
        _visible.pop(r, None)
        changed = True
    if changed:
        _legend_box.children = [_role_buttons[r] for r in sorted(_role_buttons)]


def _refresh_frame(_frame=None):
    try:
        d_engine, d_sched, roles = _parse_logs()
        now = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        _sync_legend_buttons(roles)

        stale_keys = [k for k in _lines if k[0] not in set(roles)]
        legend_dirty = set()
        for key in stale_keys:
            _lines[key].remove()
            legend_dirty.add(key[1])
            del _lines[key]

        # 每个 role 的全局 t0（取 engine 和 sched 首行最早时间）
        role_t0 = {}
        for role in roles:
            candidates = []
            if not d_engine[role].empty:
                candidates.append(pd.to_datetime(d_engine[role]['timestamp'].iloc[0]))
            if not d_sched[role].empty:
                candidates.append(pd.to_datetime(d_sched[role]['timestamp'].iloc[0]))
            role_t0[role] = min(candidates) if candidates else None

        for mi, (ax, col, y_range, src) in enumerate(_ax_list):
            for ri, role in enumerate(roles):
                df    = d_engine[role] if src == 'engine' else d_sched[role]
                color = COLORS[ri % len(COLORS)]
                key   = (role, mi)
                t0    = role_t0.get(role)

                if df.empty or col not in df.columns or t0 is None:
                    continue

                ts  = pd.to_datetime(df['timestamp'])
                x   = (ts - t0).dt.total_seconds().values
                y   = pd.to_numeric(df[col], errors='coerce').values
                msk = ~np.isnan(y)
                x, y = x[msk], y[msk]
                if len(x) == 0:
                    continue

                if key in _lines:
                    _lines[key].set_xdata(x)
                    _lines[key].set_ydata(y)
                else:
                    if src == 'sched':
                        # 每请求数据用散点（linewidth=0）
                        line, = ax.plot(x, y, color=color,
                                        linewidth=0, marker='.', markersize=2,
                                        alpha=0.6, label=role)
                    else:
                        line, = ax.plot(x, y, color=color, linewidth=1.2, label=role)
                    line.set_visible(_visible.get(role, True))
                    _lines[key] = line
                    legend_dirty.add(mi)

            ax.relim()
            ax.autoscale_view()
            if y_range:
                ax.set_ylim(y_range)
            if mi in legend_dirty or (ax.get_legend() is None and ax.get_lines()):
                vl = [l for l in ax.get_lines() if l.get_visible()]
                if vl:
                    ax.legend(handles=vl, labels=[l.get_label() for l in vl],
                              fontsize=7, loc='upper right')
                elif ax.get_legend():
                    ax.get_legend().remove()

        counts = {r: f'eng={len(d_engine[r])} sched={len(d_sched[r])}' for r in roles}
        fig.suptitle(f'vLLM PD Metrics v6 -- Last: {now}', fontsize=13, fontweight='bold')
        _status.value = f'OK | {now} | {counts}'
    except Exception as e:
        _status.value = f'Error: {e}'


# ── 控件 ─────────────────────────────────────────────────────────────────────
_btn_start = widgets.Button(description='Start',   button_style='success',
                            layout=widgets.Layout(width='80px'))
_btn_stop  = widgets.Button(description='Stop',    button_style='danger',
                            layout=widgets.Layout(width='80px'))
_btn_once  = widgets.Button(description='Refresh', button_style='info',
                            layout=widgets.Layout(width='80px'))
_interval  = widgets.IntSlider(value=10, min=5, max=300, step=5,
                               description='Interval(s):',
                               style={'description_width': '70px'},
                               layout=widgets.Layout(width='300px'))
_status    = widgets.Label(value='Idle')

_anim = None


def _on_start(_):
    global _anim
    if _anim:
        _anim.event_source.stop()
    _anim = FuncAnimation(fig, _refresh_frame,
                          interval=_interval.value * 1000,
                          cache_frame_data=False)
    fig.canvas.draw_idle()
    _status.value = 'Started...'


def _on_stop(_):
    global _anim
    if _anim:
        _anim.event_source.stop()
        _anim = None
    _status.value = 'Stopped'


def _on_once(_):
    _refresh_frame()
    fig.canvas.draw_idle()


_btn_start.on_click(_on_start)
_btn_stop.on_click(_on_stop)
_btn_once.on_click(_on_once)

_toolbar = widgets.HBox([_btn_start, _btn_stop, _btn_once, _interval, _status])

# ── 首次刷新 + 显示 ──────────────────────────────────────────────────────────
_refresh_frame()
display(widgets.VBox([_toolbar, _legend_box, fig.canvas]))